In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb
import shap

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)

In [ ]:
df = pd.read_csv("../data/processed/features.csv")

feature_cols = [
    "cumulative_energy_MJ", "time_since_heating_s",
    *[f"Bulk_{i}_cum" for i in range(1, 16)],
    "total_bulk_cum",
    *[f"Wire_{i}_cum" for i in range(1, 10)],
    "total_wire_cum",
    "gas_total", "prev_temp", "measurement_index",
]

train_df = df[~df["is_test"]].copy()
X_train = train_df[feature_cols]
y_train = train_df["Temperature"]

print(f"Training set: {X_train.shape}")

Retraining Final Model with the same tuned hyperparameters from notebook 03 and Computing SHAP Values using TreeExplainer

In [ ]:
best_params = {
    "n_estimators": 1421,
    "max_depth": 4,
    "learning_rate": 0.012172742620298122,
    "subsample": 0.6623244735373397,
    "colsample_bytree": 0.9327748010688635,
    "min_child_weight": 14,
    "reg_alpha": 0.01452444993323096,
    "reg_lambda": 0.12000051232654022,
    "random_state": 42,
    "n_jobs": -1,
}

model = xgb.XGBRegressor(**best_params)
model.fit(X_train, y_train, verbose=False)

In [ ]:
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_train)

print(f"SHAP values shape: {shap_values.shape}")
print(f"Expected value (base prediction): {explainer.expected_value:.1f} °C")

Verifying SHAP additivity: base value + sum of SHAP values ≈ prediction

In [ ]:
sample_idx = 0
pred = model.predict(X_train.iloc[[sample_idx]])[0]
shap_sum = explainer.expected_value + shap_values[sample_idx].sum()
print(f"Prediction:              {pred:.2f} °C")
print(f"Base + SHAP sum:         {shap_sum:.2f} °C")
print(f"Difference:              {abs(pred - shap_sum):.6f} °C")

 ### SHAP computation verification

 The SHAP values matrix has the expected shape — 9784 rows × 31 features,
 one attribution value per feature per training observation. The expected value
 (base prediction) is 1592.2°C, matching the global mean temperature from
 Notebook 03. This is the model's prediction in the absence of any feature
 information — the starting point from which SHAP values push each prediction
 higher or lower.

 The additivity check confirms that SHAP values are exact: the base value plus
 the sum of all feature contributions reconstructs the model prediction to within
 0.0002°C. This is a property of TreeExplainer — it computes exact Shapley values
 for tree ensembles, not approximations. Every subsequent analysis in this notebook
 rests on this guarantee: when a SHAP value says "cumulative energy pushed this
 prediction up by 15°C," that attribution is mathematically exact.

### Global feature importance

In [ ]:
shap_df = pd.DataFrame(shap_values, columns=feature_cols)

mean_abs_shap = shap_df.abs().mean().sort_values(ascending=False)
print("Mean |SHAP| per feature:")
print(mean_abs_shap.round(2).to_string())

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
mean_abs_shap.plot(kind="barh", ax=ax, edgecolor="black", alpha=0.7)
ax.set_xlabel("Mean |SHAP value| (°C)")
ax.set_title("Global Feature Importance — Mean Absolute SHAP")
ax.invert_yaxis()
plt.tight_layout()
plt.show()

Top 5 features — cumulative contribution

In [ ]:
total_shap = mean_abs_shap.sum()
cumulative = mean_abs_shap.cumsum() / total_shap * 100

print(f"\nCumulative contribution of top features:")
for i, (feat, pct) in enumerate(cumulative.items()):
    print(f"  {i+1}. {feat:30s} — {pct:.1f}%")
    if pct > 95:
        break

SHAP Summary Plot

In [ ]:
shap.summary_plot(shap_values, X_train, feature_names=feature_cols, show=False)
plt.tight_layout()
plt.show()

In [ ]:
import shap
import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np
import pandas as pd

feature_name_map = {
    "prev_temp": "Previous temperature (°C)",
    "cumulative_energy_MJ": "Arc heating energy (MJ)",
    "time_since_heating_s": "Time since last heating (s)",
    "total_bulk_cum": "Total bulk additions (kg)",
    "Bulk_14_cum": "Bulk addition 14 (kg)",
    "gas_total": "Stirring gas (Nm³)",
    "measurement_index": "Measurement sequence",
    "Bulk_4_cum": "Bulk addition 4 (kg)",
    "Wire_1_cum": "Wire addition 1 (kg)",
    "Bulk_12_cum": "Bulk addition 12 (kg)",
}

explainer = shap.TreeExplainer(model)
shap_values = explainer(X_train)

mean_abs_shap = pd.Series(
    np.abs(shap_values.values).mean(axis=0),
    index=X_train.columns
).sort_values(ascending=False)

top_features = mean_abs_shap.head(10).index.tolist()
shap_values_top = shap_values[:, top_features]

# Rename features for display
shap_values_top.feature_names = [
    feature_name_map.get(f, f) for f in top_features
]

mpl.rcParams.update({
    "font.family": "sans-serif",
    "font.size": 12,
    "axes.titlesize": 13,
    "axes.titleweight": "bold",
})

fig, ax = plt.subplots(figsize=(10, 6))
fig.patch.set_facecolor("#F8F8F8")
ax.set_facecolor("#F8F8F8")

shap.plots.beeswarm(
    shap_values_top,
    max_display=10,
    show=False,
    plot_size=None,
    color_bar_label="Feature value",
)

ax = plt.gca()
ax.set_facecolor("#F8F8F8")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.spines["left"].set_visible(False)
ax.tick_params(axis="y", labelsize=12)
ax.tick_params(axis="x", labelsize=10)
ax.set_xlabel("SHAP value — impact on predicted temperature (°C)", fontsize=11)
ax.axvline(0, color="gray", linewidth=0.8, linestyle="--", alpha=0.5)

plt.title(
    "SHAP analysis: drivers of steel temperature in the Ladle Furnace",
    fontsize=13,
    fontweight="bold",
    pad=14,
    loc="left",
)

plt.tight_layout(pad=1.5)
plt.savefig("shap_beeswarm_linkedin.png", dpi=180, bbox_inches="tight", facecolor="#F8F8F8")
plt.show()
print("Saved to shap_beeswarm_linkedin.png")

 ### Global importance structure

 The importance ranking confirms that the scoped model has learned a physically
 coherent representation of LF thermodynamics, with the removal of first
 measurements sharpening the signal in every major feature.

 | Rank | Feature | Mean SHAP (°C) | Cumulative % | Physical role |
 |---|---|---|---|---|
 | 1 | prev_temp | 9.95 | 47.5% | Current thermal state |
 | 2 | cumulative_energy_MJ | 3.80 | 65.7% | Energy input (arc heating) |
 | 3 | time_since_heating_s | 1.86 | 74.6% | Energy loss (radiation + conduction) |
 | 4 | measurement_index | 0.69 | 77.9% | Process stage |

 These four features account for 78% of total attribution. The next eight
 features bring the total to 95%, adding individual material effects, wire
 additions, and gas usage. The remaining 19 features contribute effectively
 zero — rare materials correctly suppressed even with the weaker L1
 regularization in the revised tuning.

 **Comparison with the original model:** `prev_temp` increased from 8.49 to
 9.95°C mean |SHAP| — now that every training row has this feature available
 (no more NaN from first measurements), the model relies on it more heavily.
 `cumulative_energy_MJ` rose from 2.88 to 3.80°C, reflecting its clearer role
 when the prediction problem is purely incremental: the model uses energy to
 explain how much temperature changed since the last reading, not to guess
 the absolute thermal state from scratch.

 **New material signals:** `Bulk_6_cum` (0.34°C) emerged as a distinct
 contributor — previously suppressed by the strong L1 regularization (1.60)
 in the original tuning, now visible with the weaker L1 (0.015) that the
 scoped dataset favors. `Wire_1_cum` rose to rank 5 (0.63°C), ahead of
 `total_bulk_cum` — confirming that CaSi wire injection has a per-kilogram
 thermal impact that the model distinguishes from the bulk aggregate.

 **prev_temp dominance** — the beeswarm confirms a strong monotonic positive
 relationship: high previous temperature pushes predictions up, low pushes
 them down, with SHAP values spanning approximately −40°C to +70°C. Unlike
 the original model, there is no cluster of near-zero SHAP values from NaN
 rows — every observation receives a meaningful attribution from this feature.

 **cumulative_energy_MJ** — monotonic positive, with SHAP values ranging from
 roughly −10°C to +20°C. The model correctly associates more arc energy with
 higher predicted temperature.

 **time_since_heating_s** — monotonic negative: longer elapsed time since the
 last heating session pushes predictions down, consistent with radiative and
 conductive heat loss from the ladle.

 **measurement_index** — now rank 4, up from rank 7. With first measurements
 removed, the index captures process stage more cleanly: low index (early heat)
 pushes down, high index (late heat) pushes up, reflecting the general LF
 trajectory where temperature rises through the treatment cycle.

### SHAP dependence plot

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

shap.dependence_plot("prev_temp", shap_values, X_train,
                     feature_names=feature_cols, ax=axes[0, 0], show=False)
axes[0, 0].set_title("prev_temp")

shap.dependence_plot("cumulative_energy_MJ", shap_values, X_train,
                     feature_names=feature_cols, ax=axes[0, 1], show=False)
axes[0, 1].set_title("cumulative_energy_MJ")

shap.dependence_plot("time_since_heating_s", shap_values, X_train,
                     feature_names=feature_cols, ax=axes[1, 0], show=False)
axes[1, 0].set_title("time_since_heating_s")

shap.dependence_plot("total_bulk_cum", shap_values, X_train,
                     feature_names=feature_cols, ax=axes[1, 1], show=False)
axes[1, 1].set_title("total_bulk_cum")

plt.tight_layout()
plt.show()

 ### Dependence plot interpretation

 **prev_temp** — near-linear positive relationship with approximately 0.7°C
 SHAP increase per 1°C increase in previous temperature. The tight scatter
 confirms this is the cleanest signal in the dataset. The interaction coloring
 by cumulative energy reveals a physically meaningful pattern: at the same
 previous temperature, higher cumulative energy (red) pushes predictions
 slightly higher — the model correctly compounds the thermal state anchor
 with the energy delivered since the last reading. At low previous temperatures
 (1525–1560°C), the SHAP values are strongly negative (−20 to −40°C),
 pulling predictions well below the global mean — these are early-heat
 measurements where the bath is still cold after arrival from primary refining.

 **cumulative_energy_MJ** — nonlinear with saturation. SHAP rises steeply
 from 0 to ~1000 MJ, then flattens above ~2000 MJ. This reflects the
 diminishing marginal impact of energy input at high temperatures: radiation
 losses scale with T⁴, so the hotter the bath gets, the more energy is needed
 to achieve the same temperature increment. The interaction coloring by
 measurement index is informative: blue points (early measurements, low index)
 cluster at low energy with negative SHAP, while red points (late measurements,
 high index) cluster at high energy with positive SHAP. This is the LF process
 lifecycle rendered in data — energy accumulates as the heat progresses, and
 the model tracks this accumulation correctly.

 **time_since_heating_s** — steep negative drop in the first 200–500 seconds,
 then gradual decline extending to extreme values. The model has learned an
 approximate cooling curve consistent with radiative heat loss: the initial
 rate is highest immediately after heating (when the surface temperature is
 at its peak), then decelerates as the surface cools. The interaction with
 `prev_temp` shows that hotter melts (red) lose more heat per unit time —
 as radiation physics predicts.
 Points beyond 2000 seconds represent extended waiting scenarios (caster
 delays, operational holds), where the model correctly applies increasingly
 large negative corrections.

 **total_bulk_cum** — predominantly negative with a positive bump at low
 values (0–200 kg). The negative trend is thermodynamically direct: cold
 solid material absorbs energy from the liquid steel as it heats to bath
 temperature and dissolves. The positive bump at low bulk reflects a confound
 — heats with minimal additions tend to be simple grades that arrived close
 to target, so low bulk mass co-occurs with higher temperatures without
 causing them. Above ~500 kg the relationship is clearly negative and
 approximately linear, with each additional 500 kg reducing predicted
 temperature by roughly 1°C — a magnitude consistent with the enthalpy
 balance for ferroalloy dissolution in a ~100 tonne steel ladle. The
 interaction with `prev_temp` reinforces the thermal sink interpretation:
 at the same bulk load, hotter melts (red) show slightly more positive SHAP
 because the temperature anchor partially compensates the cooling effect.

### Single-prediction explanation

Case 1: typical mid-heat measurement

In [ ]:
mid_heat_candidates = train_df[
    (train_df["measurement_index"] >= 4) &
    (train_df["Temperature"].between(1590, 1610)) &
    (train_df["Temperature"] >= train_df["prev_temp"]) &
    ((train_df["Temperature"] - train_df["prev_temp"]) <= 8)
].copy()

mid_heat = mid_heat_candidates.iloc[0]
mid_heat_idx = X_train.index.get_loc(mid_heat.name)

print(f"Case 1 — Mid-heat measurement (heat {int(mid_heat['key'])}, "
      f"index {int(mid_heat['measurement_index'])})")
print(f"  Actual:    {mid_heat['Temperature']:.0f} °C")
print(f"  Predicted: {model.predict(X_train.iloc[[mid_heat_idx]])[0]:.1f} °C")
print(f"  prev_temp: {mid_heat['prev_temp']:.0f} °C  "
      f"(delta: +{mid_heat['Temperature'] - mid_heat['prev_temp']:.0f} °C)")
print(f"  cumulative_energy_MJ: {mid_heat['cumulative_energy_MJ']:.0f}")
print(f"  total_bulk_cum: {mid_heat['total_bulk_cum']:.0f} kg")

In [ ]:
shap.waterfall_plot(
    shap.Explanation(
        values=shap_values[mid_heat_idx],
        base_values=explainer.expected_value,
        data=X_train.iloc[mid_heat_idx],
        feature_names=feature_cols,
    ),
    max_display=10,
    show=False,
)
plt.title(f"Case 1 — Mid-heat (heat {int(mid_heat['key'])})")
plt.tight_layout()
plt.show()

Case 2: second measurement — earliest prediction point in the scoped model

In [ ]:
second_meas = train_df[
    (train_df["measurement_index"] == 2) &
    (train_df["Temperature"].between(1560, 1590))
].iloc[0]
second_meas_idx = X_train.index.get_loc(second_meas.name)

print(f"Case 2 — Second measurement (heat {int(second_meas['key'])}, index 2)")
print(f"  Actual:    {second_meas['Temperature']:.0f} °C")
print(f"  Predicted: {model.predict(X_train.iloc[[second_meas_idx]])[0]:.1f} °C")
print(f"  prev_temp: {second_meas['prev_temp']:.0f} °C")
print(f"  cumulative_energy_MJ: {second_meas['cumulative_energy_MJ']:.0f} MJ")

In [ ]:
shap.waterfall_plot(
    shap.Explanation(
        values=shap_values[second_meas_idx],
        base_values=explainer.expected_value,
        data=X_train.iloc[second_meas_idx],
        feature_names=feature_cols,
    ),
    max_display=10,
    show=False,
)
plt.title(f"Case 2 — Second measurement (heat {int(second_meas['key'])})")
plt.tight_layout()
plt.show()

Case 3: measurement after heavy bulk addition

In [ ]:
heavy_bulk = train_df[
    (train_df["total_bulk_cum"] > train_df["total_bulk_cum"].quantile(0.95)) &
    (train_df["measurement_index"] >= 3)
].iloc[0]
heavy_bulk_idx = X_train.index.get_loc(heavy_bulk.name)

print(f"Case 3 — Heavy bulk addition (heat {int(heavy_bulk['key'])}, "
      f"index {int(heavy_bulk['measurement_index'])})")
print(f"  Actual:    {heavy_bulk['Temperature']:.0f} °C")
print(f"  Predicted: {model.predict(X_train.iloc[[heavy_bulk_idx]])[0]:.1f} °C")
print(f"  total_bulk_cum: {heavy_bulk['total_bulk_cum']:.0f} kg")
print(f"  prev_temp: {heavy_bulk['prev_temp']:.0f} °C")

In [ ]:
shap.waterfall_plot(
    shap.Explanation(
        values=shap_values[heavy_bulk_idx],
        base_values=explainer.expected_value,
        data=X_train.iloc[heavy_bulk_idx],
        feature_names=feature_cols,
    ),
    max_display=10,
    show=False,
)
plt.title(f"Case 3 — Heavy bulk (heat {int(heavy_bulk['key'])})")
plt.tight_layout()
plt.show()

 ## Single-prediction interpretation

 Three cases illustrate the model's behavior across distinct operating regimes.

 **Case 1 — Mid-heat (heat 2, 5th measurement, error 4.0°C):**
 A stable heat in its final treatment stage — temperature has risen 4°C from
 the previous reading (1604→1608°C), confirming a heating event occurred since
 the last measurement. The waterfall shows `prev_temp` = 1604°C delivering the
 dominant push (+8.18°C), correctly anchoring the prediction above the global
 mean. Cumulative energy (+2.56°C) adds a secondary upward correction consistent
 with 734 MJ of total arc input. Bulk 4 contributes a small positive effect
 (+0.65°C) — unusually, this pushes the prediction *up* rather than down. Most
 bulk additions act as thermal sinks, but if Bulk 4 corresponds to aluminum for
 final deoxidation, the positive direction would be physically consistent: the
 reaction 2[Al] + 3[O] → Al₂O₃ is strongly exothermic,
 releasing more energy than the sensible heat needed to bring the cold aluminum
 to bath temperature. The timing — 5th measurement, late in the heat — is also
 consistent with final deoxidation practice. Without confirmed material
 identities in this anonymized dataset, this remains a plausible interpretation
 rather than a confirmed one. The cooling features act in the expected direction:
 gas (−0.30°C) and time since heating (−0.25°C, only 306 seconds elapsed). The
 prediction of 1604.0°C against an actual of 1608°C is a 4.0°C error — within
 thermocouple uncertainty.

 **Case 2 — Second measurement (heat 2, index 2, error 6.2°C):**
 The earliest prediction point in the scoped model, and the hardest. The
 previous temperature is 1581°C — below the global mean — so `prev_temp`
 correctly pushes the prediction down (−6.28°C). But the dominant feature here
 is `cumulative_energy_MJ` at only 60 MJ, pushing the prediction down by
 −7.62°C. This is the model recognizing that very little arc energy has been
 delivered — the heat is still in its early phase, far from the temperatures
 typically seen later in treatment. The `time_since_heating_s` contribution is
 positive (+1.60°C) at 89 seconds — a short gap indicating the arc just turned
 off, so the model does not apply a large cooling correction. The net prediction
 of 1583.2°C overshoots the actual (1577°C) by 6.2°C, consistent with the
 elevated MAE at index 2 seen in the residual analysis: the initial heating
 ramp produces the largest temperature jumps between consecutive readings,
 making incremental prediction hardest at this stage.

 **Case 3 — Heavy bulk (heat 12, 3rd measurement, error 5.1°C):**
 Tests the model's response to a large thermal sink. `prev_temp` = 1606°C
 anchors the prediction above the mean (+7.85°C), but this is countered by
 two strong cooling effects: `time_since_heating_s` at 835 seconds (−5.23°C)
 — nearly 14 minutes since the last arc session, representing substantial
 radiative loss — and Bulk 12 at 618 kg (−4.42°C), a large individual material
 addition acting as a direct thermal sink. The model attributes the bulk cooling
 primarily to Bulk 12 specifically (−4.42°C) rather than to the aggregate
 `total_bulk_cum` (−0.63°C), demonstrating that it has learned material-specific
 enthalpy effects. Bulk 14 adds a further −0.78°C for its 406 kg contribution.
 The net prediction of 1590.1°C overshoots the actual (1585°C) by 5.1°C —
 the model slightly underestimates the combined cooling from 1273 kg of bulk
 additions plus 14 minutes of radiative loss, but captures the direction and
 approximate magnitude correctly.

 ## Summary

 This notebook used SHAP to explain the XGBoost model's predictions and connect
 its learned behavior to the thermodynamics and kinetics of the Ladle Furnace
 process. The model was trained on the scoped dataset — 9784 intermediate and
 final measurements where `prev_temp` is always available.

  What the model learned

 The model discovered a physically coherent representation of LF operation
 without being given any physics equations:

 1. **Temperature is incremental** — the previous measurement is the dominant
    predictor (47.5% of total attribution, mean |SHAP| = 9.95°C), correctly
    reflecting that the LF process is a sequence of small perturbations from the
    current thermal state.

 2. **Arc energy heats the steel** — cumulative energy input has a positive,
    saturating effect on predicted temperature (18.2% of attribution), consistent
    with the diminishing marginal returns of heating at elevated temperatures
    where radiation losses scale with T⁴.

 3. **Time means cooling** — elapsed time since the last heating session has a
    negative effect that follows an approximate radiative cooling curve (8.9% of
    attribution), with steep initial loss rates that decelerate over time. The
    interaction with `prev_temp` confirms that hotter melts cool faster — exactly
    as radiation physics predicts.

 4. **Additions are thermal sinks** — bulk and wire additions push predictions
    down in proportion to their cumulative mass, with the model learning
    material-specific thermal impacts. Bulk 12, Bulk 4, and Bulk 6 survive
    regularization and contribute separate SHAP effects beyond the aggregate
    total, indicating distinct enthalpies of dissolution.

 5. **Wire injection matters individually** — Wire 1 (likely CaSi, nearly
    universal) ranks 5th overall with 0.63°C mean |SHAP|, ahead of the bulk
    aggregate. The model distinguishes its per-kilogram thermal impact from
    the bulk materials.

  Operational relevance

 The SHAP analysis demonstrates that the model's learned relationships are
 physically trustworthy — its predictions move in the right direction for the
 right reasons. For an operator or process engineer evaluating whether to trust
 a model prediction, this is the critical assurance: the model is not exploiting
 statistical artifacts, it is encoding the same thermal logic that governs the
 actual process.

 The model achieves 5.71°C MAE in cross-validation and 6.56°C on the test set —
 within the noise floor of industrial temperature measurement for mid-heat
 predictions. The single-prediction waterfall plots show that even in challenging
 cases (early measurements, heavy additions), the model produces physically
 interpretable attribution breakdowns where every major contribution maps to a
 real thermodynamic mechanism.

 The clear dominance of `prev_temp` also highlights where operational data
 collection matters most: ensuring that every temperature measurement is
 accurately recorded and correctly timestamped has a larger impact on model
 performance than adding any new sensor. Data quality on the existing
 measurements is worth more than new instrumentation.